In [ ]:
import geopandas as gpd
import pandas as pd

In [ ]:
# spatial extent
agglom_extent_filepath = "../data/raw/zurich-extent.gpkg"

# AWS data (to get the hot-day periods)
aws_ts_df_filepath = "../data/interim/aws-ts-df.csv"

# UGZ data files
ugz_stations_filepath = (
    "../data/raw/ugz_stadtklima_zuerich_messorte_messnetz_meteoblue.csv"
)
ugz_ts_parquet_filepath = (
    "../data/raw/"
    "ugz_stadtklima_zuerich_temperaturmessungen_messnetz_meteoblue_2019-aktuell.parquet"
)
# AWEL data files (cached raw inputs, see `scripts/cache_awel_data.py`)
awel_ts_df_filepath = "../data/raw/awel-ts-df.csv"
awel_stations_filepath = "../data/raw/awel-stations.gpkg"
# output files
dst_ts_df_filepath = "../data/interim/lcs-ts-df.csv"
dst_stations_gdf_filepath = "../data/interim/lcs-stations.gpkg"

In [ ]:
# load the hot-day periods identified in aws-data
aws_ts_df = pd.read_csv(
    aws_ts_df_filepath, index_col=["heatwave", "time"], parse_dates=["time"]
)
# get unique dates for filtering LCS data
hot_times = aws_ts_df.index.get_level_values("time")
hot_dates = hot_times.normalize().unique()

# also build a mapping from time -> heatwave label for reindexing LCS data
time_to_heatwave = (
    aws_ts_df.reset_index("heatwave")["heatwave"].groupby(level="time").first()
)
for year in sorted(hot_dates.year.unique()):
    year_dates = hot_dates[hot_dates.year == year]
    print(
        f"  {year}: {year_dates.min().date()} to {year_dates.max().date()} "
        f"({len(year_dates)} days)"
    )
print(f"Total: {len(hot_dates)} hot days")

In [ ]:
# read UGZ parquet (covers 2019 to present)
ugz_long = pd.read_parquet(ugz_ts_parquet_filepath)

# deduplicate on (timestamp, locationID) and keep temperature only
ugz_long = (
    ugz_long[ugz_long["parameter"] == "T"]
    .drop_duplicates(subset=["timestamp", "locationID"])
    .copy()
)

# parse timestamps and filter to hot-day dates
ugz_long["timestamp"] = pd.to_datetime(ugz_long["timestamp"])
if ugz_long["timestamp"].dt.tz is not None:
    ugz_long["timestamp"] = ugz_long["timestamp"].dt.tz_convert(None)
ugz_long = ugz_long[ugz_long["timestamp"].dt.normalize().isin(hot_dates)]

# pivot to wide (time x location)
ugz_ts_df = ugz_long.pivot_table(
    index="timestamp", columns="locationID", values="value", aggfunc="mean"
)
ugz_ts_df.index.name = "time"
ugz_ts_df = ugz_ts_df.resample("h").mean()
ugz_ts_df.head()

In [ ]:
# ACHTUNG: the AWEL series is read from a cached raw input rather than queried from the
# meteora client, which derives its station roster from the most recent monthly file
# only. Sensors that have since been relocated or decommissioned are therefore dropped
# from the whole historical record, and the client's month list also skips the first
# month whenever the requested period starts mid-month. Sensors that moved during the
# study period enter the cache as one station per location.
region = gpd.read_file(agglom_extent_filepath)["geometry"].iloc[:1]
awel_stations_gdf = (
    gpd.read_file(awel_stations_filepath).set_index("station_id").to_crs(region.crs)
)
# keep the sensors within the region, using the coordinates that they had while they
# were recording
awel_stations_gdf = awel_stations_gdf[awel_stations_gdf.within(region.iloc[0])]

awel_ts_df = pd.read_csv(awel_ts_df_filepath, index_col="time", parse_dates=["time"])
awel_ts_df = awel_ts_df[awel_ts_df.columns.intersection(awel_stations_gdf.index)]
awel_ts_df = awel_ts_df[awel_ts_df.index.normalize().isin(hot_dates)]
awel_ts_df = awel_ts_df.resample("h").mean()
awel_ts_df.head()

In [ ]:
# combine UGZ and AWEL into a single time series
lcs_ts_df = pd.concat([ugz_ts_df, awel_ts_df], axis="columns")
# drop duplicate columns if any (e.g., overlapping stations)
lcs_ts_df = lcs_ts_df.loc[:, ~lcs_ts_df.columns.duplicated()]

# align to the heatwave MultiIndex from the AWS data
# map each hourly timestamp to its heatwave label
lcs_ts_df["heatwave"] = lcs_ts_df.index.map(time_to_heatwave)
lcs_ts_df = (
    lcs_ts_df.dropna(subset=["heatwave"]).reset_index().set_index(["heatwave", "time"])
)
lcs_ts_df.head()

In [ ]:
stations_crs = "epsg:2056"

# UGZ stations
ugz_stations_df = pd.read_csv(ugz_stations_filepath)
ugz_stations_gdf = gpd.GeoDataFrame(
    ugz_stations_df.set_index("locationID")[[]],
    geometry=gpd.points_from_xy(ugz_stations_df["EKoord"], ugz_stations_df["NKoord"]),
    crs=stations_crs,
)

# AWEL stations (index = station_id), already filtered to the region above
awel_stations_gdf = awel_stations_gdf[["geometry"]].to_crs(stations_crs)

# combine and keep only stations present in the time series
lcs_stations_gdf = pd.concat([ugz_stations_gdf, awel_stations_gdf]).rename_axis(
    "station_id"
)
lcs_stations_gdf = lcs_stations_gdf.loc[
    lcs_ts_df.columns.intersection(lcs_stations_gdf.index)
]
lcs_stations_gdf.head()

In [ ]:
# rename zurich kaserne (in ts_df and stations_gdf) to avoid conflicts with NABEL AWS
rename_dict = {"Zürich-Kaserne": "Zürich-Kaserne-UGZ"}
lcs_stations_gdf = lcs_stations_gdf.rename(index=rename_dict)
lcs_ts_df = lcs_ts_df.rename(columns=rename_dict)

In [ ]:
lcs_ts_df.to_csv(dst_ts_df_filepath)
lcs_stations_gdf.reset_index().to_file(dst_stations_gdf_filepath)